## Overview
This notebook explores three different extractive text summarizers:

1. **Frequency-based scoring** — score sentences by how many "important" (frequent, non-stopword) words they contain.
2. **TF-IDF-based scoring** — same idea, but weight words by how distinctive they are, not just how often they appear.
3. **TextRank** — treat sentences as nodes in a graph and rank them by similarity to each other.

Each summary is evaluated against the original article using **ROUGE**, a standard metric for comparing generated text to a reference text.

The notebook is split into two parts: a simple implementation of Frequency-based scoring (Part 1) built directly with loose variables, followed by a cleaner, reusable class-based implementation (Part 2) that all three methods share.

# Simple Token Method (Term Frequency)
**Written:** <u>June 16 - 19, 2026</u><br>

- This method focuses on retaining "important" sentences while removing the "non-important" sentences through a scoring method.


> Dataset source: [AsadMahmood (Kaggle)](https://www.kaggle.com/datasets/asad1m9a9h6mood/news-articles)

### Setup
Libraries used in this section: `pandas` to load the CSV, `re` for regex-based text cleanup, NLTK's `sent_tokenize` / `word_tokenize` to split text into sentences and words, `stopwords` to filter out common function words (*the*, *is*, *and*, etc.) that carry little meaning, `Counter` to tally word frequencies, and `nlargest` (from `heapq`) to pull out the top-scoring sentences efficiently without sorting the entire list.

In [1]:
# Libraries
import pandas as pd
import re

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from collections import Counter
from heapq import nlargest

In [2]:
# Load Dataset
df = pd.read_csv("Articles.csv", encoding="cp1252")  # file encoding difference (cp1252 works)

print(df.head())
print(df.shape)

                                             Article      Date  \
0  KARACHI: The Sindh government has decided to b...  1/1/2015   
1  HONG KONG: Asian markets started 2015 on an up...  1/2/2015   
2  HONG KONG:  Hong Kong shares opened 0.66 perce...  1/5/2015   
3  HONG KONG: Asian markets tumbled Tuesday follo...  1/6/2015   
4  NEW YORK: US oil prices Monday slipped below $...  1/6/2015   

                                             Heading  NewsType  
0  sindh govt decides to cut public transport far...  business  
1                    asia stocks up in new year trad  business  
2           hong kong stocks open 0.66 percent lower  business  
3             asian stocks sink euro near nine year   business  
4                 us oil prices slip below 50 a barr  business  
(2692, 4)


### Tokenizing a sample article
Two things happen here:

- **Regex fix:** `\.(?=[A-Z])` finds a period that's immediately followed by a capital letter with no space (e.g. `"...reported.Sources said"`) and inserts a space. This matters because `sent_tokenize` relies on that whitespace as a cue for where one sentence ends and the next begins; without it, two sentences would incorrectly be treated as one.
- **Tokenization:** `sent_tokenize` splits the article into a list of full sentences (kept for later, since we want to display whole sentences in the summary). `word_tokenize` splits the *lowercased* article into individual word/punctuation tokens, which we'll use to build a frequency table below.

The next two cells just print each result to sanity-check that the splitting worked as expected.

In [3]:
# Tokenize sample
article = df["Article"][0].strip()  # can select from index 0 to 2691
article = re.sub(r'\.(?=[A-Z])', '. ', article)  # add spaces between sentences

sentences = sent_tokenize(article)
words = word_tokenize(article.lower())

In [4]:
# Tokenize sample sentences
print(sentences)

['KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.', 'Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling.', 'Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision.', 'KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG).', 'Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.']


In [5]:
# Tokenize sample words
print(words)

['karachi', ':', 'the', 'sindh', 'government', 'has', 'decided', 'to', 'bring', 'down', 'public', 'transport', 'fares', 'by', '7', 'per', 'cent', 'due', 'to', 'massive', 'reduction', 'in', 'petroleum', 'product', 'prices', 'by', 'the', 'federal', 'government', ',', 'geo', 'news', 'reported', '.', 'sources', 'said', 'reduction', 'in', 'fares', 'will', 'be', 'applicable', 'on', 'public', 'transport', ',', 'rickshaw', ',', 'taxi', 'and', 'other', 'means', 'of', 'traveling', '.', 'meanwhile', ',', 'karachi', 'transport', 'ittehad', '(', 'kti', ')', 'has', 'refused', 'to', 'abide', 'by', 'the', 'government', 'decision', '.', 'kti', 'president', 'irshad', 'bukhari', 'said', 'the', 'commuters', 'are', 'charged', 'the', 'lowest', 'fares', 'in', 'karachi', 'as', 'compare', 'to', 'other', 'parts', 'of', 'the', 'country', ',', 'adding', 'that', '80pc', 'vehicles', 'run', 'on', 'compressed', 'natural', 'gas', '(', 'cng', ')', '.', 'bukhari', 'said', 'karachi', 'transporters', 'will', 'cut', 'fares

### Removing stopwords
Words are filtered down to the ones that actually carry meaning: `word.isalnum()` drops standalone punctuation tokens (like `":"` or `","` that `word_tokenize` keeps as separate tokens), and `word not in stop_words` drops common function words. What's left (`filtered`) is the vocabulary we'll use to score sentence importance — the intuition being that sentences packed with meaningful, frequently-repeated words are more likely to represent the article's main point.

In [6]:
# Remove stopwords
stop_words = set(stopwords.words("english"))

filtered = []

for word in words:
    if word.isalnum() and word not in stop_words:
        filtered.append(word)

In [7]:
# Check
print(filtered)

['karachi', 'sindh', 'government', 'decided', 'bring', 'public', 'transport', 'fares', '7', 'per', 'cent', 'due', 'massive', 'reduction', 'petroleum', 'product', 'prices', 'federal', 'government', 'geo', 'news', 'reported', 'sources', 'said', 'reduction', 'fares', 'applicable', 'public', 'transport', 'rickshaw', 'taxi', 'means', 'traveling', 'meanwhile', 'karachi', 'transport', 'ittehad', 'kti', 'refused', 'abide', 'government', 'decision', 'kti', 'president', 'irshad', 'bukhari', 'said', 'commuters', 'charged', 'lowest', 'fares', 'karachi', 'compare', 'parts', 'country', 'adding', '80pc', 'vehicles', 'run', 'compressed', 'natural', 'gas', 'cng', 'bukhari', 'said', 'karachi', 'transporters', 'cut', 'fares', 'decrease', 'cng', 'prices', 'made']


### Word frequency
`Counter` tallies how many times each remaining word appears in the article. Words like *"karachi"* and *"fares"* appearing most often is exactly the signal this method relies on. The more a word is repeated across the article, the more it's treated as "important" when scoring sentences next.

In [8]:
# Count word frequency
freq = Counter(filtered)

In [9]:
# Check word frequency
print(freq)

Counter({'karachi': 4, 'fares': 4, 'government': 3, 'transport': 3, 'said': 3, 'public': 2, 'reduction': 2, 'prices': 2, 'kti': 2, 'bukhari': 2, 'cng': 2, 'sindh': 1, 'decided': 1, 'bring': 1, '7': 1, 'per': 1, 'cent': 1, 'due': 1, 'massive': 1, 'petroleum': 1, 'product': 1, 'federal': 1, 'geo': 1, 'news': 1, 'reported': 1, 'sources': 1, 'applicable': 1, 'rickshaw': 1, 'taxi': 1, 'means': 1, 'traveling': 1, 'meanwhile': 1, 'ittehad': 1, 'refused': 1, 'abide': 1, 'decision': 1, 'president': 1, 'irshad': 1, 'commuters': 1, 'charged': 1, 'lowest': 1, 'compare': 1, 'parts': 1, 'country': 1, 'adding': 1, '80pc': 1, 'vehicles': 1, 'run': 1, 'compressed': 1, 'natural': 1, 'gas': 1, 'transporters': 1, 'cut': 1, 'decrease': 1, 'made': 1})


### Scoring each sentence
Each sentence gets a score equal to the sum of the frequencies of the meaningful words it contains. Concretely: for every sentence, we re-tokenize it, and for every word in it that also shows up in our `freq` table, we add that word's frequency to the sentence's running total. A sentence that repeats several high-frequency words ends up with a high score.

In [10]:
# Score each sentence
scores = {}

for sentence in sentences:  # loop through each sentence
    for word in word_tokenize(sentence.lower()):  # tokenize per sentence
        if word in freq:
            scores[sentence] = scores.get(sentence, 0) + freq[word]

In [11]:
print(scores)

{'KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.': 37, 'Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling.': 20, 'Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision.': 17, 'KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG).': 32, 'Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.': 21}


### Building the summary
`nlargest(3, scores, key=scores.get)` pulls out the 3 sentences with the highest scores, in descending order of score (not necessarily their original order in the article). Those three sentences, joined together, become the extractive summary. No new text is generated, only the "most important" original sentences are kept.

In [12]:
summary = nlargest(3, scores, key=scores.get)  # filter top 3 sentences

print("ORIGINAL:")
print(article)
print()
print("SUMMARY:")
print(" ".join(summary))

ORIGINAL:
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported. Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling. Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision. KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG). Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.

SUMMARY:
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported. KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to o

# Extractive Methods (TF-IDF and Cosine Similarity)
**Note: This portion was made with the assistance of OpenAI's ChatGPT on the coding aspect**
- Using Term Frequency, Term Frequency - Inverse Document Frequency, Text Ranking (TF-IDF + Cosine Similarity). We create a text summarizer with a simple UI.

> Dataset source: [AsadMahmood (Kaggle)](https://www.kaggle.com/datasets/asad1m9a9h6mood/news-articles)

### Additional libraries
This part rebuilds the same idea using proper vector-space methods and packages the whole thing into reusable classes:

- `numpy` — array math for summing similarity/TF-IDF scores.
- `networkx` — builds a graph out of sentences for the TextRank method.
- `sklearn.feature_extraction.text.TfidfVectorizer` — converts sentences into TF-IDF vectors (word importance weighted by rarity across the article, not just raw count).
- `sklearn.metrics.pairwise.cosine_similarity` — measures how similar two sentence vectors are.
- `rouge_score` — scores how well a summary matches a reference text (used later for evaluation).

In [13]:
# Additional Libraries
import numpy as np
import networkx as nx
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer

In [14]:
# Stop words
stop_words = set(stopwords.words("english"))

### Reusable preprocessing
Rather than repeating tokenization/cleaning logic inside each summarizer, `Preprocessor` centralizes it:

- `sentences()` — same sentence-boundary fix as Part 1, then splits into sentences.
- `clean()` — lowercases text and strips out anything that isn't a letter, digit, or whitespace (punctuation gone entirely this time, rather than filtered token-by-token like in Part 1).
- `remove_stopwords()` — filters a list of words against the stopword set.

Keeping `original` (unmodified) sentences separate from `cleaned` ones lets us score/vectorize on clean text while still displaying properly punctuated sentences in the final summary.

In [15]:
# Data preprocessing class
class Preprocessor:

    @staticmethod
    def sentences(text):
        text = re.sub(r'\.(?=[A-Z])', '. ', text)  # spaces between sentences
        text = re.sub(r'\s+', ' ', text).strip()  # cleans spacing
        return sent_tokenize(text)
    
    @staticmethod
    def clean(text):
        text = text.lower()
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # only take alphanumeric characters
        return text.strip()

    @staticmethod
    def remove_stopwords(words):
        return [word for word in words if word not in stop_words]

### A shared interface for all summarizers
`BaseSummarizer` is an abstract base class: it defines `preprocess()` once (so Frequency, TF-IDF, and TextRank all get sentence splitting/cleaning for free) and declares `summarize()` as a method that *must* be overridden by any subclass — calling it directly raises `NotImplementedError`. This is the classic template-method pattern: shared setup lives in the base class, and each subclass only needs to implement its own scoring logic.

In [16]:
# Base class
class BaseSummarizer:

    def preprocess(self, article):
        original = Preprocessor.sentences(article)  # get sentence tokens
        cleaned = [Preprocessor.clean(s) for s in original]  # clean
        return original, cleaned

    def summarize(self, article):
        raise NotImplementedError  # error catcher (no method found)

### Method 1 — Frequency (class version)
This is a direct refactor of the Part 1 approach into the shared class structure: build a word-frequency table from the cleaned article, score each sentence by summing the frequencies of the words it contains, and keep the top 3. The only functional difference from Part 1 is that stopword/punctuation removal now goes through `Preprocessor.clean()` instead of the `isalnum()` check used earlier.

In [17]:
# Simple method (word frequency per sentence)
class FrequencySummarizer(BaseSummarizer):  # similar to the imlpementation above (added a feature that retains original text with punctuations)

    def summarize(self, article):
        original, cleaned = self.preprocess(article)
        words = word_tokenize(Preprocessor.clean(article))
        words = Preprocessor.remove_stopwords(words)
        frequency = Counter(words)

        scores = {}
        
        for original_s, cleaned_s in zip(original, cleaned):
            for word in word_tokenize(cleaned_s):
                if word in frequency:
                    scores[original_s] = scores.get(original_s, 0) + frequency[word]
    
        summary = nlargest(3, scores, key=scores.get)
        return " ".join(summary)

### Method 2 — TF-IDF
Instead of scoring by raw word count, `TfidfVectorizer` scores each word by **TF-IDF**: term frequency (how often a word appears in a given sentence) multiplied by inverse document frequency (how rare that word is across *all* sentences in the article). This down-weights words that are common throughout the whole article (which raw frequency scoring would over-value) and up-weights words that are distinctive to a specific sentence. Each sentence becomes a row in the resulting matrix; summing each row gives a sentence-level importance score, and the top 3 are kept as the summary.

In [18]:
# TF-IDF (term frequency - inverse document frequency)
class TFIDFSummarizer(BaseSummarizer):  # how frequent is a word in a sentence - how rare or common is that word across the entire article

    def summarize(self, article):
        original, cleaned = self.preprocess(article)
        matrix = TfidfVectorizer().fit_transform(cleaned)  # matrix = rows:sentences and columns:words then get TF-IDF
        
        scores = np.asarray(matrix.sum(axis=1)).flatten()  # add up all the scores
        best = nlargest(3, range(len(scores)), scores.take)  # extract the top 3
    
        summary = [original[i] for i in best]
        return " ".join(summary)

### Method 3 — TextRank
This builds on TF-IDF by adding relationships *between* sentences, not just word importance within them:

1. Vectorize sentences with TF-IDF (same as Method 2).
2. Compute `cosine_similarity` between every pair of sentence vectors; how "close" each sentence is to every other sentence in meaning.
3. Treat that similarity matrix as a graph (`networkx`), where each sentence is a node and edge weights are similarity scores.
4. Run `pagerank` on the graph. Here, a sentence "ranks" highly if it's similar to many other important sentences, which tends to surface sentences that summarize the article's central themes rather than just the most word-frequent ones.

The top 3 ranked sentences become the summary.

In [19]:
# Text rank
class TextRankSummarizer(BaseSummarizer):  # same as TF-IDF with an addition of cosine similarity

    def summarize(self, article):
        original, cleaned = self.preprocess(article)
        matrix = TfidfVectorizer().fit_transform(cleaned)
        similarity = cosine_similarity(matrix)  # measures the angle between vectors (TF-IDF) in a geometric space
        graph = nx.from_numpy_array(similarity)  # converts that table into a living network graph
    
        scores = nx.pagerank(graph)
        ranked = sorted(scores, key=scores.get, reverse=True)
        
        summary = [original[i] for i in ranked[:3]]
        return " ".join(summary)

### Evaluation
To judge how good each summary is, we compare it back against the original article using **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation). This is the standard metric for extractive/abstractive summarization quality.

In [20]:
# Evaluation
class Evaluator:  # uses ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

    @staticmethod
    def rouge(reference, prediction):
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        return scorer.score(reference, prediction)

***A little more explanation on ROUGE***  
In the simplest words, rouge1 (word count - did the AI mention the right topics and keywords), rouge2 (phrasing - did the AI capture the right descriptions and context), rougeL (flow and structure - did the AI keep the correct word order and sentence structure). Higher is better for all three.

### Putting it together: an interactive comparison
This final cell wires everything into a simple command-line loop: pick an article by index, generate a summary with all three methods, score each one with ROUGE against the full article, and print the results side by side. It's a way to eyeball how the three approaches differ in practice; not a formal evaluation, since scoring a summary against the *entire* article (rather than a human-written reference summary) naturally favors longer, more literal overlaps over genuinely well-chosen sentences.

##### Static

In [21]:
# Basic UI
CSV_PATH = "Articles.csv"
df = pd.read_csv(CSV_PATH, encoding="cp1252")
df["Article"] = df["Article"].str.replace(r'\.(?=[A-Z])', '. ', regex=True)
df["Article"] = df["Article"].str.replace(r'\s+', ' ', regex=True).str.strip()

frequency = FrequencySummarizer()
tfidf = TFIDFSummarizer()
textrank = TextRankSummarizer()
evaluator = Evaluator()  # Instantiate the Evaluator

# --- Set the article index here ---
ARTICLE_INDEX = 0  # <-- change this number to pick a different article (0-2691)

if ARTICLE_INDEX < 0 or ARTICLE_INDEX >= len(df):
    print("\nInvalid index!\n")
else:
    heading = df.loc[ARTICLE_INDEX, "Heading"]
    article = df.loc[ARTICLE_INDEX, "Article"]

    # Generate Summaries
    frequency_summary = frequency.summarize(article)
    tfidf_summary = tfidf.summarize(article)
    textrank_summary = textrank.summarize(article)

    # Calculate ROUGE Scores (comparing each summary back to the original full article)
    freq_scores = evaluator.rouge(article, frequency_summary)
    tfidf_scores = evaluator.rouge(article, tfidf_summary)
    textrank_scores = evaluator.rouge(article, textrank_summary)

    print("\n")
    print("=" * 60)
    print("ORIGINAL HEADING")
    print("=" * 60)
    print(heading)
    print("\n")
    print("=" * 60)
    print("ORIGINAL ARTICLE")
    print("=" * 60)
    print(article)
    print("\n")
    print("=" * 60)
    print("LEVEL 1 - FREQUENCY")
    print("=" * 60)
    print(frequency_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {freq_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {freq_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {freq_scores['rougeL'].fmeasure:.4f}")
    print("\n")
    print("=" * 60)
    print("LEVEL 2 - TF-IDF")
    print("=" * 60)
    print(tfidf_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {tfidf_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {tfidf_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {tfidf_scores['rougeL'].fmeasure:.4f}")
    print("\n")
    print("=" * 60)
    print("LEVEL 3 - TEXTRANK")
    print("=" * 60)
    print(textrank_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {textrank_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {textrank_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {textrank_scores['rougeL'].fmeasure:.4f}")
    print()



ORIGINAL HEADING
sindh govt decides to cut public transport fares by 7pc kti rej


ORIGINAL ARTICLE
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported. Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling. Meanwhile, Karachi Transport Ittehad (KTI) has refused to abide by the government decision. KTI President Irshad Bukhari said the commuters are charged the lowest fares in Karachi as compare to other parts of the country, adding that 80pc vehicles run on Compressed Natural Gas (CNG). Bukhari said Karachi transporters will cut fares when decrease in CNG prices will be made.


LEVEL 1 - FREQUENCY
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.

##### Interactive

In [22]:
# Basic UI
CSV_PATH = "Articles.csv"
df = pd.read_csv(CSV_PATH, encoding="cp1252")
df["Article"] = df["Article"].str.replace(r'\.(?=[A-Z])', '. ', regex=True)
df["Article"] = df["Article"].str.replace(r'\s+', ' ', regex=True).str.strip()

frequency = FrequencySummarizer()
tfidf = TFIDFSummarizer()
textrank = TextRankSummarizer()
evaluator = Evaluator()  # Instantiate the Evaluator

while True:
    print("=" * 60)
    print("TEXT SUMMARIZATION COMPARISON")
    print("=" * 60)

    index = input("\nEnter article index (0-2691) or q to quit: ")

    if index.lower() == "q":
        break

    try:
        index = int(index)
        if index < 0 or index >= len(df):
            print("\nInvalid index!\n")
            continue
    except ValueError:
        print("\nPlease enter a valid number.\n")
        continue

    heading = df.loc[index, "Heading"]
    article = df.loc[index, "Article"]

    # Generate Summaries
    frequency_summary = frequency.summarize(article)
    tfidf_summary = tfidf.summarize(article)
    textrank_summary = textrank.summarize(article)

    # Calculate ROUGE Scores (comparing each summary back to the original full article)
    freq_scores = evaluator.rouge(article, frequency_summary)
    tfidf_scores = evaluator.rouge(article, tfidf_summary)
    textrank_scores = evaluator.rouge(article, textrank_summary)

    print("\n")
    print("=" * 60)
    print("ORIGINAL HEADING")
    print("=" * 60)
    print(heading)

    print("\n")
    print("=" * 60)
    print("ORIGINAL ARTICLE")
    print("=" * 60)
    print(article)

    print("\n")
    print("=" * 60)
    print("LEVEL 1 - FREQUENCY")
    print("=" * 60)
    print(frequency_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {freq_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {freq_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {freq_scores['rougeL'].fmeasure:.4f}")

    print("\n")
    print("=" * 60)
    print("LEVEL 2 - TF-IDF")
    print("=" * 60)
    print(tfidf_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {tfidf_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {tfidf_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {tfidf_scores['rougeL'].fmeasure:.4f}")

    print("\n")
    print("=" * 60)
    print("LEVEL 3 - TEXTRANK")
    print("=" * 60)
    print(textrank_summary)
    print("-" * 60)
    print(f"ROUGE-1 F1: {textrank_scores['rouge1'].fmeasure:.4f} | "
          f"ROUGE-2 F1: {textrank_scores['rouge2'].fmeasure:.4f} | "
          f"ROUGE-L F1: {textrank_scores['rougeL'].fmeasure:.4f}")
    print()

TEXT SUMMARIZATION COMPARISON



Enter article index (0-2691) or q to quit:  q


# The Math Behind Each Method

The three summarizers above all rest on one shared trick: turning text into numbers so that arithmetic and linear algebra can be applied to it. Below is a walkthrough of that shared foundation, followed by the specific math each method builds on top of it.

Note: Claude AI was used in order to help futher understand this section as well as keep it accurate.

## How to turn words into numbers

Computers can't do arithmetic on strings, so every method here starts with **vectorization**: representing text as vectors in a numeric space.

**Vocabulary indexing.** Scan the corpus (or article) and build a vocabulary where every unique word gets an index. If the vocabulary has $V$ unique words, every document lives in a $V$-dimensional space.

**Bag-of-words (BoW).** The simplest encoding: represent a sentence as a vector where position $i$ holds the count of vocabulary word $i$ in that sentence. Word order is discarded such as "dog bites man" and "man bites dog" get the same vector. This is what `Counter` is building in the Frequency method above, and what `word.isalnum()` + stopword filtering is preparing the input for.

A sentence like *"The virus spreads quickly"* isn't stored as text at all under the hood — conceptually it becomes a mostly-zero vector:

$$
[0, 0, \underbrace{1}_{\text{spreads}}, 0, \dots, \underbrace{1}_{\text{virus}}, \dots, \underbrace{1}_{\text{quickly}}, \dots, 0]
$$

This is the bridge between language and linear algebra where once text is a vector, you can add, weight, and measure angles/distances.

## 1. Frequency-based scoring

**Idea:** words that appear often in *this article* are probably central to its topic, so sentences containing many high-frequency words are probably important.

**Steps:**
1. Tokenize the article into words, remove stopwords ("the", "is", "and", ...).
2. Count occurrences of each remaining word → raw frequency $f(w)$, via `Counter`.
3. Normalize by the max frequency so scores sit in $[0, 1]$:

$$
\text{score}(w) = \frac{f(w)}{\max_{w'} f(w')}
$$

4. Score each sentence $S$ by summing the scores of the words it contains:

$$
\text{score}(S) = \sum_{w \in S} \text{score}(w)
$$

5. Rank sentences by this score and keep the top-N (`nlargest`) for the summary.

This is just arithmetic on the counts from the bag-of-words vector. No linear algebra required yet.

**Weakness:** a word like "said" or "year" can be very frequent in news articles without being meaningful.

## 2. TF-IDF-based scoring

**Idea:** don't just reward frequent words, but reward words that are frequent *in this article* but rare *elsewhere in the corpus*. That's what makes a word distinctive rather than just common.

**Term Frequency (TF)** — how often word $w$ appears in document $d$, normalized by document length:

$$
\text{TF}(w, d) = \frac{\text{count of } w \text{ in } d}{\text{total words in } d}
$$

**Inverse Document Frequency (IDF)** — how rare $w$ is across all $N$ documents in the corpus:

$$
\text{IDF}(w) = \log\left(\frac{N}{1 + \text{DF}(w)}\right)
$$

where $\text{DF}(w)$ is the number of documents containing $w$, and the $+1$ avoids division by zero. If $w$ appears in almost every document ($\text{DF}(w) \approx N$), $\text{IDF}(w) \approx \log(1) = 0$ then it gets zeroed out. If $w$ appears in only a few documents, IDF is large, so it's treated as distinctive.

**TF-IDF weight:**

$$
\text{TFIDF}(w, d) = \text{TF}(w, d) \times \text{IDF}(w)
$$

Each document (or here, each *sentence*, treated as a mini-document) becomes a vector where each dimension is a word's TF-IDF weight instead of a raw count.

**Sentence scoring** works the same way as Frequency-based scoring, just with TF-IDF weights instead of raw counts:

$$
\text{score}(S) = \sum_{w \in S} \text{TFIDF}(w, d)
$$

Rank and keep the top-N sentences.

## Worked Example: TF-IDF by Hand

Let's walk through TF-IDF on a tiny toy corpus of 3 "documents" (imagine these are 3 sentences from an article):

- **Doc 1:** `the cat sat on the mat`
- **Doc 2:** `the dog sat on the log`
- **Doc 3:** `cats and dogs are great pets`

We'll compute the TF-IDF weight for the word **"cat"** in Doc 1, step by step.

---

### Step 1 — Term Frequency (TF)

$$
\text{TF}(w, d) = \frac{\text{count of } w \text{ in } d}{\text{total words in } d}
$$

Doc 1 has 6 words total (`the, cat, sat, on, the, mat`), and "cat" appears once:

$$
\text{TF}(\text{"cat"}, \text{Doc 1}) = \frac{1}{6} \approx 0.167
$$

Compare this to "the", which appears twice in Doc 1:

$$
\text{TF}(\text{"the"}, \text{Doc 1}) = \frac{2}{6} \approx 0.333
$$

Just looking at TF, "the" looks *more* important than "cat" — which is backwards. That's the problem IDF fixes.

---

### Step 2 — Inverse Document Frequency (IDF)

$$
\text{IDF}(w) = \log\left(\frac{N}{1 + \text{DF}(w)}\right)
$$

Here $N = 3$ (total documents), and $\text{DF}(w)$ is how many documents contain $w$ at least once.

**For "cat":** appears only in Doc 1 → $\text{DF}(\text{"cat"}) = 1$

$$
\text{IDF}(\text{"cat"}) = \log\left(\frac{3}{1+1}\right) = \log(1.5) \approx 0.405
$$

**For "the":** appears in Doc 1 and Doc 2 → $\text{DF}(\text{"the"}) = 2$

$$
\text{IDF}(\text{"the"}) = \log\left(\frac{3}{1+2}\right) = \log(1) = 0
$$

"the" gets an IDF of **zero** because it shows up in almost every document, it carries no distinguishing power. This is IDF doing exactly what it's supposed to: punishing words that are common *everywhere*, regardless of how often they appear in a single document.

---

### Step 3 — Combine into TF-IDF

$$
\text{TFIDF}(w, d) = \text{TF}(w, d) \times \text{IDF}(w)
$$

$$
\text{TFIDF}(\text{"cat"}, \text{Doc 1}) = 0.167 \times 0.405 \approx 0.0676
$$

$$
\text{TFIDF}(\text{"the"}, \text{Doc 1}) = 0.333 \times 0 = 0
$$

Even though "the" appeared *more often* in Doc 1, its final TF-IDF score is **0**, while "cat", appearing just once, keeps a positive score. TF-IDF has correctly identified "cat" as the more meaningful, topic-defining word for Doc 1.

---

### Step 4 — What a full row looks like

If we ran this over the whole vocabulary, Doc 1 would become a vector where each dimension is a word's TF-IDF score:

| word | TF (Doc 1) | DF | IDF | TF-IDF (Doc 1) |
|---|---|---|---|---|
| the | 0.333 | 2 | 0.000 | 0.000 |
| cat | 0.167 | 1 | 0.405 | 0.068 |
| sat | 0.167 | 2 | 0.000 | 0.000 |
| on  | 0.167 | 2 | 0.000 | 0.000 |
| mat | 0.167 | 1 | 0.405 | 0.068 |

Doc 1's vector: `[0.000, 0.068, 0.000, 0.000, 0.068]` (in the order the table lists the words).

This is exactly what `TfidfVectorizer.fit_transform()` produces internally, one row per document (in the notebook's case, one row per **sentence**), one column per vocabulary word, filled with these weighted scores instead of raw counts. Scikit-learn's actual formula includes L2 normalization and a slightly different IDF smoothing by default, but the core idea above is identical.

---

### Step 5 — Why this matters for summarization

When scoring a sentence, we sum the TF-IDF weights of its words:

$$
\text{score}(S) = \sum_{w \in S} \text{TFIDF}(w, d)
$$

A sentence packed with words like "cat" and "mat" (rare, distinctive) scores higher than a sentence packed with words like "the" and "sat" (common across the corpus, contribute nothing), even if the second sentence has more total words. This is the mechanism that lets TF-IDF summarization surface sentences carrying the article's actual content, rather than sentences that just happen to be long or full of filler words.

## 3. TextRank

This method is the most different mathematically as it moves from "sum of word weights" to **graph theory + eigenvectors**.

**Step A — Vectorize each sentence.** Represent every sentence in the article as a TF-IDF vector (same math as above, applied at sentence granularity).

**Step B — Measure sentence similarity.** Instead of comparing words, compare *sentences to each other* using **cosine similarity**:

$$
\text{sim}(S_i, S_j) = \frac{\vec{S_i} \cdot \vec{S_j}}{\|\vec{S_i}\| \, \|\vec{S_j}\|}
$$

This is the dot product of the two vectors divided by the product of their magnitudes (geometrically), the cosine of the angle between them. Sentences that share many distinctive words point in a similar direction in that high-dimensional space, giving a small angle and a similarity close to 1. Unrelated sentences are closer to orthogonal, giving a similarity near 0.

**Step C — Build a graph.** Every sentence is a node. Draw an edge between every pair of sentences $(S_i, S_j)$, weighted by $\text{sim}(S_i, S_j)$. This produces an $n \times n$ similarity matrix $M$ ($n$ = number of sentences), which doubles as a weighted adjacency matrix for the graph.

**Step D — Rank nodes with PageRank.** This is Google's PageRank algorithm, repurposed for sentences: a sentence is "important" if it's similar to *other important sentences*, importance is recursive, so it can't be computed in a single pass; it has to be solved for as a stable equilibrium. The score for sentence $S_i$:

$$
\text{TR}(S_i) = (1-d) + d \sum_{S_j \in \text{links}(S_i)} \frac{\text{sim}(S_i, S_j)}{\sum_{S_k \in \text{links}(S_j)} \text{sim}(S_j, S_k)} \, \text{TR}(S_j)
$$

- $d$ is a damping factor (typically 0.85).
- Solved iteratively: initialize every sentence with score 1, plug into the right-hand side, get updated scores, repeat until scores stop changing much (convergence).
- Equivalently, this is finding the **dominant eigenvector** of the (normalized) similarity matrix $M$. The steady-state distribution of a random walk where you jump between sentences with probability proportional to their similarity.

**Intuition:** imagine randomly wandering between sentences, more likely to hop to a sentence if it's similar to where you currently are. The TextRank score is the fraction of time you'd spend at each sentence in the long run. Sentences that are "hubs", similar to many other central sentences, get visited most often, and those are the ones picked for the summary.

## The throughline

| Method | Representation | Scoring mechanism |
|---|---|---|
| Frequency | word counts | sum word scores |
| TF-IDF | word counts reweighted by rarity | sum word scores |
| TextRank | sentence vectors (TF-IDF-based) | graph + eigenvector (PageRank) |

Frequency and TF-IDF are essentially the same algorithm with a smarter weighting scheme plugged in. TextRank uses TF-IDF-style vectors too, but instead of scoring words directly, it uses those vectors to measure sentence-to-sentence *relationships*, then finds importance the same way Google ranks web pages by looking at what's well-connected to other important things, not just what's individually frequent.

This is also why the ROUGE evaluation used throughout this notebook fits naturally as a comparison point: ROUGE itself is just n-gram overlap counting between the summary and the reference a simpler cousin of the same "turn text into countable units, then compare" logic used above.

## The Math Behind ROUGE

ROUGE (**R**ecall-**O**riented **U**nderstudy for **G**isting **E**valuation) measures how much overlap there is between a generated summary and a reference text, by counting shared chunks of text into single words, pairs of words, or longest matching sequences. The notebook uses three variants: **ROUGE-1**, **ROUGE-2**, and **ROUGE-L**, each measuring overlap at a different granularity.

For all three, we compute the same three numbers:

$$
\text{Precision} = \frac{\text{overlap count}}{\text{count in generated summary}}
\qquad
\text{Recall} = \frac{\text{overlap count}}{\text{count in reference text}}
$$

$$
\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

Precision asks *"of what the summary said, how much was actually in the reference?"* Recall asks *"of what the reference said, how much did the summary capture?"* F1 balances the two into a single number. This is the `.fmeasure` value the notebook prints for each method.

---

### ROUGE-1: unigram overlap

ROUGE-1 compares individual words (unigrams) between the summary and the reference.

**Reference (original article, simplified):** `the cat sat on the mat`
**Summary (generated):** `the cat sat`

Unigrams in reference: `{the, cat, sat, on, the, mat}` → 6 total
Unigrams in summary: `{the, cat, sat}` → 3 total

Matching unigrams (count each match only up to how many times it appears in *both*): `the` (1 match), `cat` (1 match), `sat` (1 match) → **3 overlapping unigrams**

$$
\text{Precision} = \frac{3}{3} = 1.00 \qquad \text{Recall} = \frac{3}{6} = 0.50
$$

$$
\text{F1} = 2 \times \frac{1.00 \times 0.50}{1.00 + 0.50} = 2 \times \frac{0.50}{1.50} \approx 0.667
$$

The summary reused only words that exist in the reference (perfect precision), but only captured half of the reference's words (moderate recall) so summarizing was aggressive.

---

### ROUGE-2: bigram overlap

ROUGE-2 does the same thing, but compares consecutive word pairs (bigrams) instead of single words. This captures word *order*, not just word presence.

Bigrams in reference: `(the,cat) (cat,sat) (sat,on) (on,the) (the,mat)` → 5 total
Bigrams in summary: `(the,cat) (cat,sat)` → 2 total

Matching bigrams: `(the,cat)`, `(cat,sat)` → **2 overlapping bigrams**

$$
\text{Precision} = \frac{2}{2} = 1.00 \qquad \text{Recall} = \frac{2}{5} = 0.40
$$

$$
\text{F1} = 2 \times \frac{1.00 \times 0.40}{1.00 + 0.40} \approx 0.571
$$

ROUGE-2's score is lower than ROUGE-1's here, that's typical. Matching a whole *pair* of consecutive words is a stricter test than matching single words, since it also checks that the words appear in the right order relative to each other. A summary can score well on ROUGE-1 (right vocabulary) but poorly on ROUGE-2 (words stitched together in a different order than the original).

---

### ROUGE-L: longest common subsequence (LCS)

ROUGE-L doesn't look at fixed-size chunks (unigrams/bigrams) at all. Instead it finds the **longest common subsequence**, the longest sequence of words that appears in both texts *in the same relative order*, but not necessarily consecutively (gaps are allowed).

$$
\text{Reference: } \underline{\text{the}} \; \underline{\text{cat}} \; \underline{\text{sat}} \; \text{on} \; \text{the} \; \text{mat}
$$
$$
\text{Summary: } \; \underline{\text{the}} \; \underline{\text{cat}} \; \underline{\text{sat}}
$$

Here the LCS is `the cat sat`, length 3, matched exactly. But consider a summary that reorders or drops words in the middle, e.g. `the mat` (skipping "cat sat on the" but keeping "the...mat" in order): the LCS would still find `the mat` as a valid subsequence of the reference, length 2, even though those two words weren't adjacent in the original.

Once the LCS length is found, precision/recall/F1 are computed the same way as before, using LCS length in place of a raw overlap count:

$$
\text{Precision} = \frac{\text{LCS length}}{\text{length of summary}} \qquad \text{Recall} = \frac{\text{LCS length}}{\text{length of reference}}
$$

For our `the cat sat` example, LCS length = 3, summary length = 3, reference length = 6:

$$
\text{Precision} = \frac{3}{3} = 1.00 \qquad \text{Recall} = \frac{3}{6} = 0.50 \qquad \text{F1} \approx 0.667
$$

(Same numbers as ROUGE-1 here, because the matched words also happened to be contiguous in this simple example, that's not always the case.)

**Why LCS matters:** ROUGE-L rewards sentence structure and word ordering without requiring an *exact* run of consecutive words like ROUGE-2 does. It sits in between ROUGE-1 (too lenient — ignores order entirely) and ROUGE-2 (too strict — requires unbroken pairs) as a measure of overall sequence similarity.

---

### Why the notebook compares a summary to the *full article* (not a human-written summary)

Normally ROUGE compares a machine summary against a human-written reference summary. Here, the notebook instead compares each summary back to the **original article**. This changes the interpretation slightly:

- **Recall** measures how much of the article's content survived into the summary — naturally low, since a summary is supposed to be much shorter than the article it came from.
- **Precision** measures how much of the summary's content is traceable back to the article — naturally high for *extractive* summarizers (like all three methods here), since every summary sentence is copied verbatim from the article rather than paraphrased.

This makes ROUGE here less an "accuracy" score and more a way to compare Frequency, TF-IDF, and TextRank *against each other* — whichever method achieves a better balance of precision/recall is selecting sentences that preserve more of the article's original content, in the article's own words and roughly its own order.

---

### Summary table

| Metric | Compares | Sensitive to word order? | Typical score relative to others |
|---|---|---|---|
| ROUGE-1 | single words | No | Highest (easiest to match) |
| ROUGE-2 | consecutive word pairs | Yes (strict, must be adjacent) | Lowest (hardest to match) |
| ROUGE-L | longest ordered subsequence | Yes (loose, gaps allowed) | Middle ground |


# Full Pipeline Example (TF-IDF → TextRank → ROUGE)

To see how these three pieces connect, let's try it on a sample article (not from the dataset) and do the whole process by hand: TF-IDF vectorizes the sentences, TextRank uses those vectors to pick the most central sentences, and ROUGE scores the resulting summary against the original article.

**Toy article (4 sentences):**

| ID | Sentence |
|---|---|
| S1 | cats sleep often |
| S2 | dogs play often |
| S3 | cats dogs pets |
| S4 | pets need care |

Vocabulary (8 words): `cats, sleep, often, dogs, play, pets, need, care`

---

### Part 1 — TF-IDF: vectorize each sentence

Treating each sentence as its own "document," $N = 4$. Every sentence has 3 words, so every word present in a sentence has $\text{TF} = 1/3$.

$$
\text{IDF}(w) = \log\left(\frac{N}{1+\text{DF}(w)}\right)
$$

| word | DF | IDF |
|---|---|---|
| cats | 2 | $\log(4/3) \approx 0.288$ |
| sleep | 1 | $\log(4/2) \approx 0.693$ |
| often | 2 | $\log(4/3) \approx 0.288$ |
| dogs | 2 | $\log(4/3) \approx 0.288$ |
| play | 1 | $\log(4/2) \approx 0.693$ |
| pets | 2 | $\log(4/3) \approx 0.288$ |
| need | 1 | $\log(4/2) \approx 0.693$ |
| care | 1 | $\log(4/2) \approx 0.693$ |

Multiplying $\text{TF} \times \text{IDF}$ for each word gives every sentence a vector (columns ordered as `cats, sleep, often, dogs, play, pets, need, care`):

$$
\begin{aligned}
\vec{S_1} &= [0.096,\ 0.231,\ 0.096,\ 0,\ 0,\ 0,\ 0,\ 0] \\
\vec{S_2} &= [0,\ 0,\ 0.096,\ 0.096,\ 0.231,\ 0,\ 0,\ 0] \\
\vec{S_3} &= [0.096,\ 0,\ 0,\ 0.096,\ 0,\ 0.096,\ 0,\ 0] \\
\vec{S_4} &= [0,\ 0,\ 0,\ 0,\ 0,\ 0.096,\ 0.231,\ 0.231]
\end{aligned}
$$

These four vectors are TF-IDF's output and they're also exactly what TextRank needs as its input.

---

### Part 2 — TextRank: rank the sentences

**Step A — Cosine similarity between every pair of sentences**, using the vectors above:

$$
\text{sim}(S_i, S_j) = \frac{\vec{S_i} \cdot \vec{S_j}}{\|\vec{S_i}\|\,\|\vec{S_j}\|}
$$

| pair | shared word | similarity |
|---|---|---|
| S1–S2 | often | 0.128 |
| S1–S3 | cats | 0.207 |
| S1–S4 | (none) | 0.000 |
| S2–S3 | dogs | 0.207 |
| S2–S4 | (none) | 0.000 |
| S3–S4 | pets | 0.163 |

**Step B — Build the graph.** These six numbers form a symmetric similarity matrix, each sentence is a node, each similarity value is an edge weight. Notice S3 ("cats dogs pets") is the only sentence connected to every other sentence as it's the bridge between the "cats/sleep" topic, the "dogs/play" topic, and the "pets/care" topic.

**Step C — Apply PageRank.** Starting every sentence at $\text{TR} = 1$, with damping $d = 0.85$:

$$
\text{TR}(S_i) = (1-d) + d\sum_{S_j \in \text{links}(S_i)} \frac{\text{sim}(S_i,S_j)}{\sum_k \text{sim}(S_j,S_k)}\,\text{TR}(S_j)
$$

Working through one iteration for $S_3$ (connected to $S_1$, $S_2$, and $S_4$):

$$
\text{TR}(S_3) = 0.15 + 0.85\left[\frac{0.207}{0.335}(1) + \frac{0.207}{0.335}(1) + \frac{0.163}{0.163}(1)\right] \approx 2.05
$$

Repeating the same calculation for the others gives, after this first pass:

| sentence | TR score |
|---|---|
| S3 | 2.05 |
| S1 | 0.78 |
| S2 | 0.78 |
| S4 | 0.39 |

Running more iterations refines these numbers further, but the ranking stabilizes here: **S3 > S1 ≈ S2 > S4**. This matches intuition, S3 touches every topic in the article, so it's treated as the most "central" sentence, exactly like a well-linked page in PageRank.

**Selecting the summary:** picking the top 2 scoring sentences gives **S3** and **S1** (S2 is a near-tie with S1 but S1 appears first in the article). Re-ordering by their original position for readability:

> **Generated summary:** "cats sleep often. cats dogs pets."

---

### Part 3 — ROUGE: score the summary against the article

**Reference** (full article, all 4 sentences concatenated): `cats sleep often dogs play often cats dogs pets pets need care` — 12 words
**Summary:** `cats sleep often cats dogs pets` — 6 words

**ROUGE-1 (unigram overlap).** Count each word only as many times as it appears in *both* texts (clipped counts):

| word | in summary | in reference | overlap |
|---|---|---|---|
| cats | 2 | 2 | 2 |
| sleep | 1 | 1 | 1 |
| often | 1 | 2 | 1 |
| dogs | 1 | 2 | 1 |
| pets | 1 | 2 | 1 |

Total overlap = 6

$$
\text{Precision} = \frac{6}{6} = 1.00 \qquad \text{Recall} = \frac{6}{12} = 0.50 \qquad \text{F1} \approx 0.667
$$

**ROUGE-2 (bigram overlap).** Summary bigrams: `(cats,sleep) (sleep,often) (often,cats) (cats,dogs) (dogs,pets)` — all 5 also appear in the reference's bigram list.

$$
\text{Precision} = \frac{5}{5} = 1.00 \qquad \text{Recall} = \frac{5}{11} \approx 0.455 \qquad \text{F1} \approx 0.625
$$

**ROUGE-L (longest common subsequence).** Checking the reference in order — `cats(1) sleep(2) often(3) dogs(4) play(5) often(6) cats(7) dogs(8) pets(9) pets(10) need(11) care(12)` — the entire summary sequence `cats, sleep, often, cats, dogs, pets` appears in order at reference positions 1, 2, 3, 7, 8, 9. So LCS length = 6 (the whole summary).

$$
\text{Precision} = \frac{6}{6} = 1.00 \qquad \text{Recall} = \frac{6}{12} = 0.50 \qquad \text{F1} \approx 0.667
$$

---

### Reading the results together

| Metric | F1 |
|---|---|
| ROUGE-1 | 0.667 |
| ROUGE-2 | 0.625 |
| ROUGE-L | 0.667 |

Precision is a perfect 1.00 across the board, expected, since this is an *extractive* summary, so every word in it is copied straight from the article. Recall sits around 0.45–0.50 for all three, which makes sense: the summary is exactly half the length of the article (6 of 12 words), so it can capture at most about half of the article's content no matter how good the sentence selection is.

Recap: **TF-IDF** turned raw sentences into vectors → **TextRank** used those vectors to find the most "connected" sentences and built a summary → **ROUGE** measured how much of the original article's content survived into that summary.